# Disease Terms Review And Source Hierarchy Modules

This notebook runs the disease-term class search across BioPortal, OLS, and OBO Foundry direct sources, writes the merged review CSV, and then builds source-specific hierarchy modules.

BioPortal modules use BioPortal class, ancestor, and parent endpoints. OLS modules use OLS term and parent endpoints. OBO Foundry Direct modules parse explicit named classes and named superclass links from downloaded OBO Foundry source files when possible.

The modules keep target classes, available superhierarchy, direct subclass edges among retrieved classes, labels, definitions, comments, and synonyms. They do not preserve full source OWL axioms.


## 1. Imports And Configuration

The default CSV output is `Scoping review/disease_terms_review.csv` relative to the notebook working directory. Check `ENV_FILE` if the BioPortal key lives somewhere else. For OLS tunnel setups, edit `OLS_BASE_URL_NOTEBOOK` in the run cell.

In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import re
import sys
import time
import urllib.parse
import xml.etree.ElementTree as ET
from datetime import datetime
from pathlib import Path
from typing import Any

import requests

# -- Config --------------------------------------------------------------------

ENV_FILE = Path(
    "/Users/Strubbelig/Library/Mobile Documents/com~apple~CloudDocs/Forschung/Ontology/Wound ontology/.env"
)

BIOPORTAL_BASE_URL = "https://data.bioontology.org"
OLS_BASE_URL = "https://www.ebi.ac.uk/ols4"
OBO_REGISTRY_URL = "https://obofoundry.org/registry/ontologies.yml"
def default_scoping_review_dir() -> Path:
    if "__file__" in globals():
        return Path(__file__).resolve().parent
    cwd = Path.cwd()
    return cwd if cwd.name == "Scoping review" else cwd / "Scoping review"


SCOPING_REVIEW_DIR = default_scoping_review_dir()
DEFAULT_OUT = SCOPING_REVIEW_DIR / "disease_terms_review.csv"

CACHE_DIR = Path("/tmp/disease_terms_review_cache")

DEFAULT_TERMS = [
    "disease",
    "disorder",
    "injury",
    "illness",
    "condition",
    "syndrome",
    "pathological process",
    "pathological disposition",
    "dysfunction",
    "predisposition",
]

BFO_PREFIX = "http://purl.obolibrary.org/obo/BFO_"
DELAY = 0.12

DEFINITION_KEYS = {
    "definition",
    "definitions",
    "description",
    "dc:description",
    "dcterms:description",
    "iao_0000115",
    "obo:iao_0000115",
    "skos:definition",
    "http://purl.obolibrary.org/obo/iao_0000115",
    "http://www.w3.org/2004/02/skos/core#definition",
}
COMMENT_KEYS = {
    "comment",
    "comments",
    "rdfs:comment",
    "skos:note",
    "skos:scopeNote",
    "scopeNote",
    "editor note",
    "iao_0000116",
    "http://www.w3.org/2000/01/rdf-schema#comment",
    "http://www.w3.org/2004/02/skos/core#note",
    "http://www.w3.org/2004/02/skos/core#scopeNote",
    "http://purl.obolibrary.org/obo/iao_0000116",
}
VALUE_KEYS = {"value", "values", "text", "literal", "label"}

## 2. Generic Helpers

In [ ]:
# -- Generic helpers -----------------------------------------------------------

def now_iso() -> str:
    return datetime.now().astimezone().isoformat(timespec="seconds")


def clean_list(values: Any) -> list[str]:
    if not values:
        return []
    if isinstance(values, str):
        values = [values]
    return [str(v).strip() for v in values if str(v).strip()]


def first_clean(values: Any) -> str:
    values = clean_list(values)
    return values[0] if values else ""


def join_values(values: Any) -> str:
    seen: set[str] = set()
    out: list[str] = []
    for value in clean_list(values):
        if value not in seen:
            out.append(value)
            seen.add(value)
    return "; ".join(out)


def append_unique(existing: str, value: str) -> str:
    parts = clean_list(existing.split(";")) if existing else []
    if value and value not in parts:
        parts.append(value)
    return "; ".join(parts)


def cache_path(namespace: str, url: str, params: dict[str, Any] | None = None) -> Path:
    params = params or {}
    serialised = json.dumps([url, sorted(params.items())], sort_keys=True, default=str)
    digest = hashlib.sha256(serialised.encode("utf-8")).hexdigest()
    path = CACHE_DIR / namespace
    path.mkdir(parents=True, exist_ok=True)
    return path / f"{digest}.json"


def get_json(
    url: str,
    params: dict[str, Any] | None = None,
    *,
    namespace: str = "generic",
    retries: int = 3,
    headers: dict[str, str] | None = None,
    cache: bool = True,
) -> Any | None:
    path = cache_path(namespace, url, params)
    if cache and path.exists():
        return json.loads(path.read_text(encoding="utf-8"))

    for attempt in range(retries):
        try:
            time.sleep(DELAY)
            response = requests.get(url, params=params or {}, headers=headers, timeout=60)
            if response.status_code == 429:
                wait = int(response.headers.get("Retry-After", 10))
                print(f"  [rate limit] waiting {wait}s", file=sys.stderr)
                time.sleep(wait)
                continue
            if response.status_code in {400, 404}:
                print(f"  [warn] GET {url} returned {response.status_code}; skipping", file=sys.stderr)
                return None
            response.raise_for_status()
            data = response.json()
            if cache:
                path.write_text(json.dumps(data), encoding="utf-8")
            return data
        except Exception as exc:
            print(f"  [warn] GET {url} attempt {attempt + 1} failed: {exc}", file=sys.stderr)
            time.sleep(2 ** attempt)
    return None


def get_text(
    url: str,
    *,
    namespace: str,
    retries: int = 3,
    cache: bool = True,
) -> str | None:
    path = cache_path(namespace, url)
    text_path = path.with_suffix(".txt")
    if cache and text_path.exists():
        return text_path.read_text(encoding="utf-8", errors="replace")

    for attempt in range(retries):
        try:
            time.sleep(DELAY)
            response = requests.get(url, timeout=120)
            if response.status_code == 429:
                wait = int(response.headers.get("Retry-After", 10))
                print(f"  [rate limit] waiting {wait}s", file=sys.stderr)
                time.sleep(wait)
                continue
            response.raise_for_status()
            text = response.text
            if cache:
                text_path.write_text(text, encoding="utf-8")
            return text
        except Exception as exc:
            print(f"  [warn] GET {url} attempt {attempt + 1} failed: {exc}", file=sys.stderr)
            time.sleep(2 ** attempt)
    return None


def bfo_anchor(parents: list[dict[str, str]]) -> str:
    for parent in parents:
        iri = parent.get("@id", "") or ""
        if iri.startswith(BFO_PREFIX):
            label = (parent.get("prefLabel") or "").strip()
            return f"{label} ({iri.split('BFO_')[1]})" if label else iri
    return ""


def obo_curie_to_iri(curie: str) -> str:
    return f"http://purl.obolibrary.org/obo/{curie.replace(':', '_')}"


def annotation_key_matches(key: str, wanted_keys: set[str]) -> bool:
    lowered = key.strip().lower()
    local = lowered.rsplit("#", 1)[-1].rsplit("/", 1)[-1]
    return lowered in wanted_keys or local in wanted_keys


def clean_annotation_value(value: Any) -> list[str]:
    if isinstance(value, dict):
        for key in VALUE_KEYS:
            if key in value:
                return clean_list(value[key])
        return []
    return clean_list(value)


def extract_annotation_values(data: Any, wanted_keys: set[str]) -> list[str]:
    values: list[str] = []
    if not isinstance(data, dict):
        return values

    for key, value in data.items():
        if annotation_key_matches(str(key), wanted_keys):
            values.extend(clean_annotation_value(value))

    for container_key in ["properties", "annotation", "annotations"]:
        container = data.get(container_key)
        if isinstance(container, dict):
            for key, value in container.items():
                if annotation_key_matches(str(key), wanted_keys):
                    values.extend(clean_annotation_value(value))
        elif isinstance(container, list):
            for item in container:
                if not isinstance(item, dict):
                    continue
                item_keys = clean_list(
                    item.get("property")
                    or item.get("predicate")
                    or item.get("annotationProperty")
                    or item.get("iri")
                    or item.get("@id")
                )
                if any(annotation_key_matches(key, wanted_keys) for key in item_keys):
                    for value_key in VALUE_KEYS:
                        values.extend(clean_annotation_value(item.get(value_key)))
                values.extend(extract_annotation_values(item, wanted_keys))

    return values


def make_row(
    *,
    term: str,
    source: str,
    ontology: str,
    ontology_version: str = "",
    preferred_label: str = "",
    class_iri: str = "",
    definition: str = "",
    comments: str = "",
    synonyms: str = "",
    parent_labels: str = "",
    parent_iris: str = "",
    bfo_anchor_value: str = "",
    mapping_count: str | int = "",
    source_url: str = "",
    retrieved_at: str | None = None,
) -> dict[str, Any]:
    retrieved_at = retrieved_at or now_iso()
    row = {
        "term": term,
        "ontology": ontology,
        "preferred_label": preferred_label,
        "class_iri": class_iri,
        "definition": definition,
        "comments": comments,
        "synonyms": synonyms,
        "parent_labels": parent_labels,
        "parent_iris": parent_iris,
        "bfo_anchor": bfo_anchor_value,
        "mapping_count": mapping_count,
        "source_databases": source,
        "retrieved_at_by_database": f"{source}: {retrieved_at}",
        "ontology_versions": f"{source}: {ontology_version}" if ontology_version else "",
        "source_ontology_ids": f"{source}: {ontology}" if ontology else "",
        "source_urls": f"{source}: {source_url}" if source_url else "",
        "source_records": json.dumps(
            [
                {
                    "source": source,
                    "retrieved_at": retrieved_at,
                    "ontology": ontology,
                    "ontology_version": ontology_version,
                    "source_url": source_url,
                }
            ],
            ensure_ascii=False,
        ),
    }
    return row


def dedupe_key(row: dict[str, Any]) -> tuple[str, str]:
    iri = str(row.get("class_iri", "")).strip()
    if iri:
        return ("iri", iri)
    label_key = "|".join(
        [
            str(row.get("ontology", "")).lower(),
            str(row.get("preferred_label", "")).lower(),
            str(row.get("term", "")).lower(),
        ]
    )
    return ("fallback", label_key)


def merge_row(existing: dict[str, Any], incoming: dict[str, Any]) -> None:
    fill_if_blank = [
        "ontology",
        "preferred_label",
        "class_iri",
        "synonyms",
        "parent_labels",
        "parent_iris",
        "bfo_anchor",
        "mapping_count",
    ]
    for field in fill_if_blank:
        if not existing.get(field) and incoming.get(field) not in (None, ""):
            existing[field] = incoming[field]

    for field in ["definition", "comments"]:
        for part in clean_list(str(incoming.get(field, "")).split(";")):
            existing[field] = append_unique(str(existing.get(field, "")), part)

    for field in [
        "source_databases",
        "retrieved_at_by_database",
        "ontology_versions",
        "source_ontology_ids",
        "source_urls",
    ]:
        for part in clean_list(str(incoming.get(field, "")).split(";")):
            existing[field] = append_unique(str(existing.get(field, "")), part)

    try:
        existing_records = json.loads(existing.get("source_records", "[]") or "[]")
        incoming_records = json.loads(incoming.get("source_records", "[]") or "[]")
        seen = {(r.get("source"), r.get("ontology"), r.get("retrieved_at")) for r in existing_records}
        for record in incoming_records:
            key = (record.get("source"), record.get("ontology"), record.get("retrieved_at"))
            if key not in seen:
                existing_records.append(record)
                seen.add(key)
        existing["source_records"] = json.dumps(existing_records, ensure_ascii=False)
    except json.JSONDecodeError:
        existing["source_records"] = incoming.get("source_records", existing.get("source_records", ""))


def merge_rows(rows_by_key: dict[tuple[str, str], dict[str, Any]], rows: list[dict[str, Any]]) -> None:
    for row in rows:
        key = dedupe_key(row)
        if key in rows_by_key:
            merge_row(rows_by_key[key], row)
        else:
            rows_by_key[key] = row

## 3. BioPortal Source

BioPortal runs first when selected. By default, raw BioPortal results are filtered to exact preferred-label matches. The extractor fetches class details to improve definitions/comments/synonyms.

In [ ]:
# -- BioPortal -----------------------------------------------------------------

def load_bioportal_api_key() -> str:
    key = os.environ.get("BIOPORTAL_KEY", "")
    if not key and ENV_FILE.exists():
        for line in ENV_FILE.read_text(encoding="utf-8").splitlines():
            if line.startswith("BIOPORTAL_KEY="):
                key = line.split("=", 1)[1].strip()
    if not key:
        sys.exit("ERROR: BIOPORTAL_KEY not found")
    return key


def bioportal_get(url: str, params: dict[str, Any], apikey: str) -> Any | None:
    request_params = {**params, "apikey": apikey}
    return get_json(url, request_params, namespace="bioportal")


def search_bioportal_exact(term: str, apikey: str) -> list[dict[str, Any]]:
    hits: list[dict[str, Any]] = []
    page = 1
    while True:
        data = bioportal_get(
            f"{BIOPORTAL_BASE_URL}/search",
            {
                "q": term,
                "require_exact_match": "true",
                "include": "prefLabel,definition,synonym,notation,properties",
                "pagesize": 100,
                "page": page,
            },
            apikey,
        )
        if not data:
            break
        hits.extend(data.get("collection", []))
        if not data.get("links", {}).get("nextPage"):
            break
        page += 1
    return hits


def fetch_bioportal_parents(ontology: str, class_iri: str, apikey: str) -> list[dict[str, Any]]:
    encoded = urllib.parse.quote(class_iri, safe="")
    data = bioportal_get(
        f"{BIOPORTAL_BASE_URL}/ontologies/{ontology}/classes/{encoded}/parents",
        {"include": "prefLabel"},
        apikey,
    )
    return data if isinstance(data, list) else []


def fetch_bioportal_mapping_count(ontology: str, class_iri: str, apikey: str) -> int:
    encoded = urllib.parse.quote(class_iri, safe="")
    data = bioportal_get(
        f"{BIOPORTAL_BASE_URL}/ontologies/{ontology}/classes/{encoded}/mappings",
        {},
        apikey,
    )
    if isinstance(data, list):
        return len(data)
    if isinstance(data, dict):
        return int(data.get("totalCount", len(data.get("collection", []))))
    return 0


def fetch_bioportal_class_details(ontology: str, class_iri: str, apikey: str) -> dict[str, Any]:
    encoded = urllib.parse.quote(class_iri, safe="")
    data = bioportal_get(
        f"{BIOPORTAL_BASE_URL}/ontologies/{ontology}/classes/{encoded}",
        {"include": "prefLabel,definition,synonym,notation,properties,cui,semanticType"},
        apikey,
    )
    return data if isinstance(data, dict) else {}


def fetch_bioportal_ontology_version(ontology: str, apikey: str) -> str:
    data = bioportal_get(f"{BIOPORTAL_BASE_URL}/ontologies/{ontology}/latest_submission", {}, apikey)
    if not isinstance(data, dict):
        return ""
    parts = []
    for key in ["version", "released", "creationDate", "submissionId"]:
        value = data.get(key)
        if value:
            parts.append(f"{key}={value}")
    return ", ".join(parts)


def bioportal_rows(
    terms: list[str],
    apikey: str,
    *,
    fetch_parents: bool,
    fetch_mappings: bool,
    exact_label_only: bool,
) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    version_cache: dict[str, str] = {}

    for term in terms:
        print(f"\n-- BioPortal: searching '{term}' --", file=sys.stderr)
        hits = search_bioportal_exact(term, apikey)
        raw_count = len(hits)
        if exact_label_only:
            term_lower = term.casefold()
            hits = [hit for hit in hits if str(hit.get("prefLabel", "")).casefold() == term_lower]
        if exact_label_only:
            print(f"   {len(hits)} exact-label hits ({raw_count} raw BioPortal hits)", file=sys.stderr)
        else:
            print(f"   {len(hits)} hits", file=sys.stderr)

        for hit in hits:
            retrieved_at = now_iso()
            iri = hit.get("@id", "")
            ontology = hit.get("links", {}).get("ontology", "").rstrip("/").split("/")[-1]
            label = hit.get("prefLabel", "").strip()
            detail = fetch_bioportal_class_details(ontology, iri, apikey) if ontology and iri else {}
            definition = join_values(
                clean_list(hit.get("definition"))
                + clean_list(detail.get("definition"))
                + extract_annotation_values(hit, DEFINITION_KEYS)
                + extract_annotation_values(detail, DEFINITION_KEYS)
            )
            comments = join_values(extract_annotation_values(hit, COMMENT_KEYS) + extract_annotation_values(detail, COMMENT_KEYS))
            synonyms = join_values(clean_list(hit.get("synonym")) + clean_list(detail.get("synonym")))

            parent_labels = parent_iris = anchor = ""
            mapping_count: str | int = ""

            if ontology and ontology not in version_cache:
                version_cache[ontology] = fetch_bioportal_ontology_version(ontology, apikey)

            if fetch_parents and ontology and iri:
                parents = fetch_bioportal_parents(ontology, iri, apikey)
                parent_labels = join_values(p.get("prefLabel") for p in parents)
                parent_iris = join_values(p.get("@id") for p in parents)
                anchor = bfo_anchor([{"@id": p.get("@id", ""), "prefLabel": p.get("prefLabel", "")} for p in parents])
                if fetch_mappings:
                    mapping_count = fetch_bioportal_mapping_count(ontology, iri, apikey)

            rows.append(
                make_row(
                    term=term,
                    source="BioPortal",
                    ontology=ontology,
                    ontology_version=version_cache.get(ontology, ""),
                    preferred_label=label,
                    class_iri=iri,
                    definition=definition,
                    comments=comments,
                    synonyms=synonyms,
                    parent_labels=parent_labels,
                    parent_iris=parent_iris,
                    bfo_anchor_value=anchor,
                    mapping_count=mapping_count,
                    source_url=f"{BIOPORTAL_BASE_URL}/ontologies/{ontology}/classes/{urllib.parse.quote(iri, safe='')}" if ontology and iri else "",
                    retrieved_at=retrieved_at,
                )
            )
            note = f" [BFO: {anchor}]" if anchor else ""
            print(f"   {ontology:15s} {label}{note}", file=sys.stderr)
    return rows

## 4. OLS Source

OLS uses `/ols4/api/search` with label-scoped exact class search and then fetches term details to capture definitions/comments from annotations where available.

In [ ]:
# -- OLS -----------------------------------------------------------------------

def normalise_ols_base_url(base_url: str) -> str:
    base_url = base_url.rstrip("/")
    return base_url if base_url.endswith("/ols4") else f"{base_url}/ols4"


def ols_get(base_url: str, path: str, params: dict[str, Any] | None = None) -> Any | None:
    return get_json(f"{base_url.rstrip('/')}{path}", params or {}, namespace="ols")


def ols_iri_variants(iri: str) -> list[str]:
    once = urllib.parse.quote(iri, safe="")
    slash_double_encoded = once.replace("%2F", "%252F")
    twice = urllib.parse.quote(once, safe="")
    variants = []
    for value in [slash_double_encoded, once, twice]:
        if value not in variants:
            variants.append(value)
    return variants


def search_ols_exact(term: str, base_url: str) -> list[dict[str, Any]]:
    docs: list[dict[str, Any]] = []
    rows = 100
    start = 0

    while True:
        data = ols_get(
            base_url,
            "/api/search",
            {
                "q": term,
                "exact": "true",
                "type": "class",
                "queryFields": "label",
                "fieldList": "iri,label,ontology_name,ontology_prefix,description,comment,annotation,synonym,exact_synonyms,obo_id,type",
                "rows": rows,
                "start": start,
            },
        )
        response = data.get("response", {}) if isinstance(data, dict) else {}
        batch = response.get("docs", [])
        docs.extend(batch)
        num_found = int(response.get("numFound", len(docs)))
        start += rows
        if start >= num_found or not batch:
            break

    term_lower = term.casefold()
    return [doc for doc in docs if str(doc.get("label", "")).casefold() == term_lower]


def fetch_ols_ontology_version(ontology: str, base_url: str) -> str:
    data = ols_get(base_url, f"/api/ontologies/{ontology}")
    if not isinstance(data, dict):
        return ""
    config = data.get("config") or {}
    parts = []
    for key in ["version", "versionIri", "title", "namespace"]:
        value = config.get(key) or data.get(key)
        if value:
            parts.append(f"{key}={value}")
    return ", ".join(parts)


def extract_ols_terms(data: Any) -> list[dict[str, Any]]:
    if isinstance(data, list):
        return data
    if not isinstance(data, dict):
        return []
    embedded = data.get("_embedded", {})
    for key in ["terms", "classes"]:
        if isinstance(embedded.get(key), list):
            return embedded[key]
    if isinstance(data.get("terms"), list):
        return data["terms"]
    if data.get("iri") or data.get("label"):
        return [data]
    return []


def fetch_ols_parents(ontology: str, iri: str, base_url: str) -> list[dict[str, str]]:
    for encoded in ols_iri_variants(iri):
        data = ols_get(base_url, f"/api/ontologies/{ontology}/terms/{encoded}/parents")
        terms = extract_ols_terms(data)
        if terms:
            return [
                {
                    "@id": parent.get("iri", ""),
                    "prefLabel": parent.get("label", ""),
                }
                for parent in terms
            ]
    return []


def fetch_ols_term_details(ontology: str, iri: str, base_url: str) -> dict[str, Any]:
    for encoded in ols_iri_variants(iri):
        data = ols_get(base_url, f"/api/ontologies/{ontology}/terms/{encoded}")
        if isinstance(data, dict) and data:
            return data
    return {}


def ols_rows(
    terms: list[str],
    *,
    base_url: str,
    fetch_parents: bool,
) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    version_cache: dict[str, str] = {}

    for term in terms:
        print(f"\n-- OLS: searching '{term}' --", file=sys.stderr)
        hits = search_ols_exact(term, base_url)
        print(f"   {len(hits)} hits", file=sys.stderr)

        for hit in hits:
            retrieved_at = now_iso()
            iri = hit.get("iri", "")
            ontology = (hit.get("ontology_name") or hit.get("ontology_prefix") or "").lower()
            label = str(hit.get("label", "")).strip()
            detail = fetch_ols_term_details(ontology, iri, base_url) if ontology and iri else {}
            definition = join_values(
                clean_list(hit.get("description"))
                + clean_list(detail.get("description"))
                + extract_annotation_values(hit, DEFINITION_KEYS)
                + extract_annotation_values(detail, DEFINITION_KEYS)
            )
            comments = join_values(
                clean_list(hit.get("comment"))
                + clean_list(detail.get("comment"))
                + extract_annotation_values(hit, COMMENT_KEYS)
                + extract_annotation_values(detail, COMMENT_KEYS)
            )
            synonyms = join_values(clean_list(hit.get("synonym")) + clean_list(hit.get("exact_synonyms")))

            if ontology and ontology not in version_cache:
                version_cache[ontology] = fetch_ols_ontology_version(ontology, base_url)

            parent_labels = parent_iris = anchor = ""
            if fetch_parents and ontology and iri:
                parents = fetch_ols_parents(ontology, iri, base_url)
                parent_labels = join_values(p.get("prefLabel") for p in parents)
                parent_iris = join_values(p.get("@id") for p in parents)
                anchor = bfo_anchor(parents)

            rows.append(
                make_row(
                    term=term,
                    source="OLS",
                    ontology=ontology,
                    ontology_version=version_cache.get(ontology, ""),
                    preferred_label=label,
                    class_iri=iri,
                    definition=definition,
                    comments=comments,
                    synonyms=synonyms,
                    parent_labels=parent_labels,
                    parent_iris=parent_iris,
                    bfo_anchor_value=anchor,
                    source_url=f"{base_url}/api/ontologies/{ontology}/terms/{ols_iri_variants(iri)[0]}" if ontology and iri else f"{base_url}/api/search",
                    retrieved_at=retrieved_at,
                )
            )
            print(f"   {ontology:15s} {label}", file=sys.stderr)
    return rows

## 5. OBO Foundry Direct Source

This scans the OBO Foundry registry directly, preferring `.obo` products and falling back to OWL/RDF XML labels. When BioPortal is also selected, OBO ontologies already indexed in BioPortal are skipped.

In [ ]:
# -- OBO Foundry direct scan ---------------------------------------------------

def find_ontology_product_urls(record: dict[str, Any]) -> list[str]:
    urls: list[str] = []

    def add_url(url: str) -> None:
        if url and url not in urls:
            urls.append(url)

    for product in record.get("products", []) or []:
        url = product.get("ontology_purl") or product.get("url") or product.get("id") or ""
        if str(url).endswith(".obo"):
            add_url(str(url))
    for product in record.get("products", []) or []:
        url = product.get("ontology_purl") or product.get("url") or product.get("id") or ""
        if str(url).endswith(".owl"):
            add_url(str(url))
    url = record.get("ontology_purl") or ""
    if str(url).endswith((".obo", ".owl")):
        add_url(str(url))
    return urls


def fetch_obo_registry() -> list[dict[str, Any]]:
    text = get_text(OBO_REGISTRY_URL, namespace="obo-registry")
    if not text:
        return []
    return parse_obo_registry_yaml(text)


def parse_simple_yaml_value(value: str) -> str:
    value = value.strip()
    if value in {"", "|", ">"}:
        return ""
    return value.strip("\"'")


def parse_obo_registry_yaml(text: str) -> list[dict[str, Any]]:
    """Extract the small subset of OBO registry YAML needed for direct scans."""
    records: list[dict[str, Any]] = []
    chunks = re.split(r"\n(?=- activity_status:)", text)
    for chunk in chunks:
        chunk = chunk.strip("\n")
        if not chunk.startswith("- activity_status:"):
            continue

        record: dict[str, Any] = {"products": []}
        lines = chunk.splitlines()
        in_license = False
        for line in lines:
            if line.startswith("- activity_status:"):
                record["activity_status"] = parse_simple_yaml_value(line.split(":", 1)[1])
                in_license = False
            elif match := re.match(r"^  (id|ontology_purl|title|preferredPrefix|domain|homepage|repository|description):\s*(.*)$", line):
                record[match.group(1)] = parse_simple_yaml_value(match.group(2))
                in_license = False
            elif line.startswith("  license:"):
                record["license"] = {}
                in_license = True
            elif in_license and (match := re.match(r"^    (label|url):\s*(.*)$", line)):
                record["license"][match.group(1)] = parse_simple_yaml_value(match.group(2))
            elif line.startswith("  ") and not line.startswith("    "):
                in_license = False

        products_block = re.search(r"(?ms)^  products:\n(.*?)(?=^  [A-Za-z0-9_-]+:|\Z)", chunk)
        if products_block:
            current_product: dict[str, str] | None = None
            for line in products_block.group(1).splitlines():
                if line.startswith("  - "):
                    if current_product:
                        record["products"].append(current_product)
                    current_product = {}
                    rest = line[4:]
                    if ":" in rest:
                        key, value = rest.split(":", 1)
                        current_product[key.strip()] = parse_simple_yaml_value(value)
                elif current_product is not None and (match := re.match(r"^    (id|ontology_purl|title):\s*(.*)$", line)):
                    current_product[match.group(1)] = parse_simple_yaml_value(match.group(2))
            if current_product:
                record["products"].append(current_product)

        records.append(record)
    return records


def parse_obo_header(text: str) -> dict[str, str]:
    header: dict[str, str] = {}
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        if line == "[Term]":
            break
        if ":" in line:
            key, value = line.split(":", 1)
            header[key.strip()] = value.strip()
    return header


def parse_obo_stanza(stanza: dict[str, list[str]], terms_lower: set[str]) -> dict[str, Any] | None:
    names = clean_list(stanza.get("name"))
    if not names:
        return None
    name = names[0]
    if name.casefold() not in terms_lower:
        return None
    if first_clean(stanza.get("is_obsolete")).lower() == "true":
        return None

    curie = first_clean(stanza.get("id"))
    definition = ""
    if stanza.get("def"):
        match = re.search(r'"(.*?)"', stanza["def"][0])
        definition = match.group(1) if match else stanza["def"][0]
    comments = join_values(stanza.get("comment"))

    synonyms = []
    for synonym_line in stanza.get("synonym", []):
        match = re.search(r'"(.*?)"', synonym_line)
        if match:
            synonyms.append(match.group(1))

    parents: list[dict[str, str]] = []
    for parent_line in stanza.get("is_a", []):
        parent_curie = parent_line.split()[0]
        parent_label = parent_line.split("!", 1)[1].strip() if "!" in parent_line else ""
        parents.append({"@id": obo_curie_to_iri(parent_curie), "prefLabel": parent_label})

    return {
        "term": name,
        "preferred_label": name,
        "class_iri": obo_curie_to_iri(curie) if curie else "",
        "definition": definition,
        "comments": comments,
        "synonyms": join_values(synonyms),
        "parent_labels": join_values(p.get("prefLabel") for p in parents),
        "parent_iris": join_values(p.get("@id") for p in parents),
        "bfo_anchor": bfo_anchor(parents),
    }


def scan_obo_text(text: str, terms: list[str]) -> tuple[str, list[dict[str, Any]]]:
    header = parse_obo_header(text)
    version = header.get("data-version") or header.get("date") or header.get("format-version") or ""
    terms_lower = {term.casefold() for term in terms}

    rows: list[dict[str, Any]] = []
    current_kind = ""
    current: dict[str, list[str]] = {}

    def flush_current() -> None:
        if current_kind != "Term" or not current:
            return
        parsed = parse_obo_stanza(current, terms_lower)
        if parsed:
            rows.append(parsed)

    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line or line.startswith("!"):
            continue
        if line.startswith("[") and line.endswith("]"):
            flush_current()
            current_kind = line.strip("[]")
            current = {}
            continue
        if current_kind and ":" in line:
            key, value = line.split(":", 1)
            current.setdefault(key.strip(), []).append(value.strip())
    flush_current()
    return version, rows


def local_name(tag: str) -> str:
    return tag.rsplit("}", 1)[-1] if "}" in tag else tag


def child_texts(element: ET.Element, names: set[str]) -> list[str]:
    values = []
    for child in element:
        if local_name(child.tag) in names and child.text and child.text.strip():
            values.append(child.text.strip())
    return values


def scan_owl_xml_text(text: str, terms: list[str]) -> tuple[str, list[dict[str, Any]]]:
    try:
        root = ET.fromstring(text.encode("utf-8"))
    except ET.ParseError as exc:
        print(f"  [warn] OWL/XML parse failed: {exc}", file=sys.stderr)
        return "", []

    rdf_about = "{http://www.w3.org/1999/02/22-rdf-syntax-ns#}about"
    rdf_resource = "{http://www.w3.org/1999/02/22-rdf-syntax-ns#}resource"
    owl_version_iri = "{http://www.w3.org/2002/07/owl#}versionIRI"
    owl_version_info = "{http://www.w3.org/2002/07/owl#}versionInfo"
    terms_lower = {term.casefold() for term in terms}

    version = ""
    for ontology in root.iter("{http://www.w3.org/2002/07/owl#}Ontology"):
        for child in ontology:
            if child.tag == owl_version_iri and child.attrib.get(rdf_resource):
                version = child.attrib[rdf_resource]
                break
            if child.tag == owl_version_info and child.text:
                version = child.text.strip()
                break
        if version:
            break

    rows: list[dict[str, Any]] = []
    class_elements = list(root.iter("{http://www.w3.org/2002/07/owl#}Class"))
    for element in class_elements:
        iri = element.attrib.get(rdf_about, "")
        if not iri:
            continue
        label = first_clean(child_texts(element, {"label"}))
        if not label or label.casefold() not in terms_lower:
            continue

        definitions = child_texts(element, {"IAO_0000115", "definition"})
        comments = child_texts(element, {"comment"})
        synonyms = []
        for child in element:
            name = local_name(child.tag)
            if "Synonym" in name and child.text and child.text.strip():
                synonyms.append(child.text.strip())

        parent_iris = []
        for child in element:
            if local_name(child.tag) == "subClassOf" and child.attrib.get(rdf_resource):
                parent_iris.append(child.attrib[rdf_resource])
        parents = [{"@id": parent_iri, "prefLabel": ""} for parent_iri in parent_iris]

        rows.append(
            {
                "term": label,
                "preferred_label": label,
                "class_iri": iri,
                "definition": first_clean(definitions),
                "comments": join_values(comments),
                "synonyms": join_values(synonyms),
                "parent_labels": "",
                "parent_iris": join_values(parent_iris),
                "bfo_anchor": bfo_anchor(parents),
            }
        )
    return version, rows


def scan_ontology_text(text: str, terms: list[str]) -> tuple[str, list[dict[str, Any]]]:
    stripped = text.lstrip()
    if stripped.startswith("format-version:") or "\n[Term]" in stripped[:5000]:
        return scan_obo_text(text, terms)
    if stripped.startswith("<?xml") or stripped.startswith("<rdf:RDF") or stripped.startswith("<Ontology"):
        return scan_owl_xml_text(text, terms)
    print("  [warn] ontology format not supported for direct OBO Foundry scan", file=sys.stderr)
    return "", []


def fetch_bioportal_ontology_ids(apikey: str) -> set[str]:
    """Return lowercased acronyms of all ontologies indexed in BioPortal."""
    data = get_json(
        f"{BIOPORTAL_BASE_URL}/ontologies",
        {"apikey": apikey, "display_links": "false", "display_context": "false"},
        namespace="bioportal",
    )
    if not isinstance(data, list):
        return set()
    ids = {str(o.get("acronym", "")).lower() for o in data if o.get("acronym")}
    print(f"   BioPortal indexes {len(ids)} ontologies — OBO scan will skip these", file=sys.stderr)
    return ids


def obo_rows(
    terms: list[str],
    *,
    max_ontologies: int | None,
    skip_ids: set[str] | None = None,
) -> list[dict[str, Any]]:
    print("\n-- OBO Foundry: loading registry --", file=sys.stderr)
    registry = fetch_obo_registry()
    active_records = [
        record for record in registry
        if str(record.get("activity_status", "active")).lower() not in {"inactive", "obsolete"}
    ]
    if skip_ids:
        before = len(active_records)
        active_records = [r for r in active_records if str(r.get("id", "")).lower() not in skip_ids]
        print(f"   skipped {before - len(active_records)} ontologies already indexed in BioPortal", file=sys.stderr)
    if max_ontologies is not None:
        active_records = active_records[:max_ontologies]
    print(f"   scanning {len(active_records)} OBO ontologies not in BioPortal", file=sys.stderr)

    rows: list[dict[str, Any]] = []
    for record in active_records:
        ontology = str(record.get("id", "")).lower()
        urls = find_ontology_product_urls(record)
        if not ontology or not urls:
            continue
        text = None
        url = ""
        for candidate_url in urls:
            print(f"   OBO {ontology}: {candidate_url}", file=sys.stderr)
            text = get_text(candidate_url, namespace="obo-files")
            if text:
                url = candidate_url
                break
        if not text:
            continue
        version, parsed_rows = scan_ontology_text(text, terms)
        retrieved_at = now_iso()
        for parsed in parsed_rows:
            rows.append(
                make_row(
                    term=parsed["term"],
                    source="OBO Foundry Direct",
                    ontology=ontology,
                    ontology_version=version,
                    preferred_label=parsed["preferred_label"],
                    class_iri=parsed["class_iri"],
                    definition=parsed["definition"],
                    comments=parsed["comments"],
                    synonyms=parsed["synonyms"],
                    parent_labels=parsed["parent_labels"],
                    parent_iris=parsed["parent_iris"],
                    bfo_anchor_value=parsed["bfo_anchor"],
                    source_url=url,
                    retrieved_at=retrieved_at,
                )
            )
    return rows

## 6. Review Enrichment

Adds ontology metadata, conservative preasserted ontology-level exclusion flags, native/imported-term guesses, and manual-screening hints. These are review aids, not final decisions.

In [ ]:
# -- Review enrichment ---------------------------------------------------------

ENRICHMENT_FIELDNAMES = [
    "ontology_title",
    "ontology_description",
    "ontology_domain",
    "ontology_homepage",
    "ontology_repository",
    "ontology_license",
    "ontology_status",
    "ontology_metadata_source",
    "ontology_metadata_retrieved_at",
    "ontology_scope_category",
    "ontology_exclusion_status",
    "ontology_exclusion_rule",
    "ontology_exclusion_reason",
    "native_ontology_guess",
    "native_term_status",
    "manual_screening_hint",
]


def source_records(row: dict[str, Any]) -> list[dict[str, Any]]:
    try:
        records = json.loads(row.get("source_records", "") or "[]")
    except json.JSONDecodeError:
        records = []
    return records if isinstance(records, list) else []


def row_source_ontology_ids(row: dict[str, Any]) -> set[str]:
    ids = {str(row.get("ontology", "")).lower()} if row.get("ontology") else set()
    for record in source_records(row):
        ontology = record.get("ontology")
        if ontology:
            ids.add(str(ontology).lower())
    return {ontology for ontology in ids if ontology}


def normalise_bioportal_ontology_metadata(data: dict[str, Any], *, retrieved_at: str | None = None) -> dict[str, str]:
    retrieved_at = retrieved_at or now_iso()
    title = first_clean([data.get("name"), data.get("acronym")])
    description = join_values(clean_list(data.get("summary")) + clean_list(data.get("description")))
    homepage = first_clean(data.get("homepage"))
    license_value = first_clean(data.get("license"))
    status = first_clean([data.get("status"), data.get("viewingRestriction")])
    domain = join_values(
        clean_list(data.get("group"))
        + clean_list(data.get("groups"))
        + clean_list(data.get("category"))
        + clean_list(data.get("categories"))
    )
    return {
        "ontology_title": title,
        "ontology_description": description,
        "ontology_domain": domain,
        "ontology_homepage": homepage,
        "ontology_repository": "",
        "ontology_license": license_value,
        "ontology_status": status,
        "ontology_metadata_source": "BioPortal ontology metadata",
        "ontology_metadata_retrieved_at": retrieved_at,
    }


def fetch_bioportal_ontology_metadata_index(apikey: str) -> dict[str, dict[str, str]]:
    data = bioportal_get(
        f"{BIOPORTAL_BASE_URL}/ontologies",
        {"display_links": "false", "display_context": "false"},
        apikey,
    )
    if not isinstance(data, list):
        return {}
    retrieved_at = now_iso()
    index = {}
    for item in data:
        if not isinstance(item, dict):
            continue
        acronym = item.get("acronym")
        if acronym:
            index[str(acronym).lower()] = normalise_bioportal_ontology_metadata(item, retrieved_at=retrieved_at)
    return index


def obo_registry_metadata_by_id() -> dict[str, dict[str, str]]:
    records = fetch_obo_registry()
    metadata: dict[str, dict[str, str]] = {}
    for record in records:
        ontology_id = str(record.get("id", "")).lower()
        if not ontology_id:
            continue
        license_value = ""
        license_data = record.get("license")
        if isinstance(license_data, dict):
            license_value = first_clean([license_data.get("label"), license_data.get("url")])
        elif license_data:
            license_value = str(license_data)
        metadata[ontology_id] = {
            "ontology_title": first_clean([record.get("title"), record.get("preferredPrefix"), ontology_id]),
            "ontology_description": first_clean(record.get("description")),
            "ontology_domain": first_clean(record.get("domain")),
            "ontology_homepage": first_clean(record.get("homepage")),
            "ontology_repository": first_clean(record.get("repository")),
            "ontology_license": license_value,
            "ontology_status": first_clean(record.get("activity_status")),
            "ontology_metadata_source": "OBO Foundry registry",
            "ontology_metadata_retrieved_at": now_iso(),
        }
    return metadata


def ontology_text(row: dict[str, Any]) -> str:
    parts = [
        row.get("ontology", ""),
        row.get("ontology_title", ""),
        row.get("ontology_description", ""),
        row.get("ontology_homepage", ""),
        row.get("ontology_repository", ""),
        row.get("source_ontology_ids", ""),
    ]
    return " ".join(str(part) for part in parts if part).lower()


def infer_ontology_scope_category(row: dict[str, Any]) -> str:
    text = ontology_text(row)
    ontology = row.get("ontology", "").lower()
    if re.search(r"\b(demo|tutorial|training|example|sample|student|thesis)\b", text):
        return "demo_or_training_resource"
    if "generated-ontology" in text or "generated ontology" in text:
        return "generated_or_local_ontology"
    if "value set" in text or ontology in {"valuesets"}:
        return "value_set_or_collection"
    if ontology in {"snomedct", "loinc", "mesh", "rh-mesh", "icd10", "icd9cm", "icpc2p", "ndfrt", "medgen"}:
        return "clinical_or_biomedical_terminology"
    if ontology in {"ncit", "doid", "mondo", "efo", "ogms", "obi", "sio"}:
        return "curated_biomedical_ontology"
    if "obo foundry" in row.get("ontology_metadata_source", "").lower():
        return "obo_foundry_ontology"
    return "ontology_or_terminology_candidate"


def preassert_exclusion(row: dict[str, Any]) -> tuple[str, str, str]:
    # OBO Foundry ontologies are accepted at the ontology level by standing policy.
    if (
        row.get("ontology_scope_category") == "obo_foundry_ontology"
        or str(row.get("class_iri", "")).startswith("http://purl.obolibrary.org/obo/")
        or "OBO Foundry Direct" in str(row.get("source_databases", ""))
    ):
        return ("candidate", "", "")
    text = ontology_text(row)
    status = row.get("ontology_status", "").lower()
    ontology = row.get("ontology", "").lower()
    if any(token in status for token in ["obsolete", "inactive", "deprecated", "retired"]):
        return ("excluded_preasserted", "E1_status_noncurrent", "Ontology metadata indicates obsolete/inactive/deprecated/retired status.")
    if re.search(r"\b(demo|tutorial|training|example|sample|student|thesis)\b", text):
        return ("excluded_preasserted", "E2_demo_training_or_thesis", "Ontology appears to be a demo, training, student, sample, or thesis resource.")
    if "generated-ontology" in text or "generated ontology" in text:
        return ("excluded_preasserted", "E3_generated_local_ontology", "Ontology appears to be generated/local rather than a reusable published ontology.")
    if "value set" in text or ontology in {"valuesets"}:
        return ("excluded_preasserted", "E4_value_set_collection", "Resource appears to be a value-set collection rather than an ontology implementation.")
    return ("candidate", "", "")


def infer_native_ontology_guess(class_iri: str) -> str:
    iri = class_iri or ""
    patterns = [
        (r"purl\.obolibrary\.org/obo/([A-Za-z0-9]+)_", None),
        (r"ebi\.ac\.uk/efo/(EFO)_", None),
        (r"ncicb\.nci\.nih\.gov/.+#C[0-9]+", "ncit"),
        (r"bioontology\.org/ontology/LNC/", "loinc"),
        (r"snomed\.info/id/", "snomedct"),
        (r"semanticscience\.org/resource/", "sio"),
        (r"w3id\.org/biolink", "biolink"),
    ]
    for pattern, fixed in patterns:
        match = re.search(pattern, iri, re.IGNORECASE)
        if match:
            return (fixed or match.group(1)).lower()
    return ""


def infer_native_term_status(row: dict[str, Any]) -> tuple[str, str]:
    native_guess = infer_native_ontology_guess(row.get("class_iri", ""))
    ontology_ids = row_source_ontology_ids(row)
    if not native_guess:
        return "", "unknown_native_ontology"
    if native_guess in ontology_ids:
        if len(ontology_ids) > 1:
            return native_guess, "native_source_seen_with_imports"
        return native_guess, "native_source_seen"
    return native_guess, "likely_imported_or_reused_term"


def manual_screening_hint(row: dict[str, Any]) -> str:
    hints = []
    if row.get("ontology_exclusion_status") == "excluded_preasserted":
        hints.append("preasserted ontology-level exclusion; verify before final exclusion")
    if not row.get("definition"):
        hints.append("missing definition")
    if not row.get("comments"):
        hints.append("missing comment")
    if row.get("native_term_status") == "likely_imported_or_reused_term":
        hints.append("term IRI namespace suggests imported/reused class")
    if row.get("ontology_scope_category") == "clinical_or_biomedical_terminology":
        hints.append("terminology/resource rather than strict ontology; decide scope")
    return "; ".join(hints)


def enrich_rows(rows: list[dict[str, Any]], *, apikey: str | None = None) -> list[dict[str, Any]]:
    apikey = apikey or load_bioportal_api_key()
    obo_metadata = obo_registry_metadata_by_id()
    bioportal_metadata = fetch_bioportal_ontology_metadata_index(apikey)

    enriched = []
    for row in rows:
        row = dict(row)
        ontology = row.get("ontology", "")
        ontology_key = ontology.lower()
        metadata = {}
        if ontology_key in obo_metadata:
            metadata.update(obo_metadata[ontology_key])
        if ontology_key in bioportal_metadata:
            for key, value in bioportal_metadata[ontology_key].items():
                if value:
                    metadata[key] = value

        for field in ENRICHMENT_FIELDNAMES:
            row.setdefault(field, "")
        row.update({key: value for key, value in metadata.items() if key in ENRICHMENT_FIELDNAMES})

        row["ontology_scope_category"] = infer_ontology_scope_category(row)
        status, rule, reason = preassert_exclusion(row)
        row["ontology_exclusion_status"] = status
        row["ontology_exclusion_rule"] = rule
        row["ontology_exclusion_reason"] = reason
        native_guess, native_status = infer_native_term_status(row)
        row["native_ontology_guess"] = native_guess
        row["native_term_status"] = native_status
        row["manual_screening_hint"] = manual_screening_hint(row)
        enriched.append(row)
    return enriched


def refresh_row_class_annotations(row: dict[str, Any], *, apikey: str, ols_base_url: str = OLS_BASE_URL) -> dict[str, Any]:
    row = dict(row)
    class_iri = row.get("class_iri", "")
    if not class_iri:
        return row

    bioportal_ontologies = []
    if "BioPortal" in row.get("source_databases", "") and row.get("ontology"):
        bioportal_ontologies.append(row["ontology"])
    for record in source_records(row):
        if record.get("source") == "BioPortal" and record.get("ontology"):
            bioportal_ontologies.append(str(record["ontology"]))
    bioportal_ontologies = list(dict.fromkeys(bioportal_ontologies))[:3]

    definitions = clean_list(str(row.get("definition", "")).split(";"))
    comments = clean_list(str(row.get("comments", "")).split(";"))
    synonyms = clean_list(str(row.get("synonyms", "")).split(";"))

    for ontology in bioportal_ontologies:
        detail = fetch_bioportal_class_details(ontology, class_iri, apikey)
        definitions.extend(clean_list(detail.get("definition")) + extract_annotation_values(detail, DEFINITION_KEYS))
        comments.extend(extract_annotation_values(detail, COMMENT_KEYS))
        synonyms.extend(clean_list(detail.get("synonym")))

    for record in source_records(row):
        if record.get("source") != "OLS" or not record.get("ontology"):
            continue
        detail = fetch_ols_term_details(str(record["ontology"]), class_iri, normalise_ols_base_url(ols_base_url))
        definitions.extend(clean_list(detail.get("description")) + extract_annotation_values(detail, DEFINITION_KEYS))
        comments.extend(clean_list(detail.get("comment")) + extract_annotation_values(detail, COMMENT_KEYS))
        synonyms.extend(clean_list(detail.get("synonym")) + clean_list(detail.get("exact_synonyms")))

    row["definition"] = join_values(definitions)
    row["comments"] = join_values(comments)
    row["synonyms"] = join_values(synonyms)
    return row


def refresh_class_annotations(rows: list[dict[str, Any]], *, ols_base_url: str = OLS_BASE_URL) -> list[dict[str, Any]]:
    apikey = load_bioportal_api_key()
    refreshed = []
    for index, row in enumerate(rows, start=1):
        if index % 25 == 0:
            print(f"  refreshed annotations for {index}/{len(rows)} rows", file=sys.stderr)
        refreshed.append(refresh_row_class_annotations(row, apikey=apikey, ols_base_url=ols_base_url))
    return refreshed


def read_csv_rows(path: Path) -> list[dict[str, Any]]:
    with path.open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))


def write_csv_rows(path: Path, rows: list[dict[str, Any]], fieldnames: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def enrich_existing_csv(in_path: Path, out_path: Path, *, refresh_annotations: bool = False) -> None:
    rows = read_csv_rows(in_path)
    if refresh_annotations:
        rows = refresh_class_annotations(rows)
    fieldnames = list(dict.fromkeys(list(rows[0].keys() if rows else FIELDNAMES) + ENRICHMENT_FIELDNAMES))
    enriched = enrich_rows(rows)
    write_csv_rows(out_path, enriched, fieldnames)
    excluded = sum(row.get("ontology_exclusion_status") == "excluded_preasserted" for row in enriched)
    print(f"Enriched {len(enriched)} rows -> {out_path} ({excluded} preasserted ontology exclusions)")

## 7. Merge And Output Helpers

In [ ]:
# -- Main ----------------------------------------------------------------------

FIELDNAMES = [
    "term",
    "ontology",
    "preferred_label",
    "class_iri",
    "definition",
    "comments",
    "synonyms",
    "parent_labels",
    "parent_iris",
    "bfo_anchor",
    "mapping_count",
    "source_databases",
    "retrieved_at_by_database",
    "ontology_versions",
    "source_ontology_ids",
    "source_urls",
    "source_records",
]


def load_existing_rows(path: Path) -> dict[tuple[str, str], dict[str, Any]]:
    rows_by_key: dict[tuple[str, str], dict[str, Any]] = {}
    if not path.exists():
        return rows_by_key
    with path.open(encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            for field in FIELDNAMES:
                row.setdefault(field, "")
            rows_by_key[dedupe_key(row)] = row
    return rows_by_key


def parse_sources(raw_sources: str) -> list[str]:
    allowed = {"bioportal", "ols", "obo"}
    sources = [source.strip().lower() for source in raw_sources.split(",") if source.strip()]
    unknown = [source for source in sources if source not in allowed]
    if unknown:
        sys.exit(f"ERROR: unknown source(s): {', '.join(unknown)}")
    if "bioportal" not in sources:
        print("[warn] BioPortal is not selected; add-on sources will run without the primary pass.", file=sys.stderr)
    return sources

## 8. Run Three-Database Review

Edit the parameters in this cell before running. The default output goes into the `Scoping review/` folder and includes enrichment columns.

In [ ]:
# Run parameters - edit these before running the cell.
OUT_PATH = SCOPING_REVIEW_DIR / "disease_terms_review_enriched.csv"
SOURCES = "bioportal,ols,obo"  # options: bioportal, ols, obo
TERMS = DEFAULT_TERMS
OLS_BASE_URL_NOTEBOOK = os.environ.get("OLS_BASE_URL", OLS_BASE_URL)
OBO_MAX_ONTOLOGIES = None  # set to an integer for a small test run
FETCH_PARENTS = True
FETCH_MAPPINGS = True
INCLUDE_NONLABEL_MATCHES = False
RESUME = False
ENRICH_OUTPUT = True


def run_review(
    *,
    out_path: Path | str = OUT_PATH,
    sources: str | list[str] = SOURCES,
    terms: list[str] = TERMS,
    ols_base_url: str = OLS_BASE_URL_NOTEBOOK,
    obo_max_ontologies: int | None = OBO_MAX_ONTOLOGIES,
    fetch_parents: bool = FETCH_PARENTS,
    fetch_mappings: bool = FETCH_MAPPINGS,
    include_nonlabel_matches: bool = INCLUDE_NONLABEL_MATCHES,
    resume: bool = RESUME,
    enrich_output: bool = ENRICH_OUTPUT,
) -> list[dict[str, Any]]:
    source_text = ",".join(sources) if isinstance(sources, list) else sources
    selected_sources = parse_sources(source_text)
    selected_terms = terms or DEFAULT_TERMS
    out_path = Path(out_path)
    rows_by_key = load_existing_rows(out_path) if resume else {}

    if rows_by_key:
        print(f"[resume] loaded {len(rows_by_key)} existing merged rows", file=sys.stderr)

    for source in selected_sources:
        if source == "bioportal":
            apikey = load_bioportal_api_key()
            merge_rows(
                rows_by_key,
                bioportal_rows(
                    selected_terms,
                    apikey,
                    fetch_parents=fetch_parents,
                    fetch_mappings=fetch_mappings,
                    exact_label_only=not include_nonlabel_matches,
                ),
            )
        elif source == "ols":
            normalised_ols_base_url = normalise_ols_base_url(ols_base_url)
            merge_rows(
                rows_by_key,
                ols_rows(selected_terms, base_url=normalised_ols_base_url, fetch_parents=fetch_parents),
            )
        elif source == "obo":
            bp_ids = fetch_bioportal_ontology_ids(load_bioportal_api_key()) if "bioportal" in selected_sources else set()
            merge_rows(rows_by_key, obo_rows(selected_terms, max_ontologies=obo_max_ontologies, skip_ids=bp_ids))

    output_rows = sorted(
        rows_by_key.values(),
        key=lambda r: (r.get("term", ""), r.get("ontology", ""), r.get("class_iri", "")),
    )
    output_fieldnames = FIELDNAMES
    if enrich_output:
        output_rows = enrich_rows(output_rows)
        output_fieldnames = FIELDNAMES + ENRICHMENT_FIELDNAMES
    write_csv_rows(out_path, output_rows, output_fieldnames)
    print(f"Done. {len(output_rows)} merged rows -> {out_path}")
    return output_rows


# Run the three-database review. For a quick smoke test, set TERMS = ["disease"], SOURCES = "ols", FETCH_PARENTS = False.
rows = run_review()
rows[:3]

## 9. Enrich Existing CSV

Use this if you already generated the raw/exact-match CSV and want to add metadata/exclusion columns or refresh class-level definitions/comments without repeating the full search.

In [ ]:
# Optional: enrich an existing CSV without rerunning the search.
RUN_ENRICH_EXISTING_CSV = False
INPUT_CSV = DEFAULT_OUT
OUTPUT_CSV = SCOPING_REVIEW_DIR / "disease_terms_review_enriched.csv"
REFRESH_CLASS_ANNOTATIONS = True

if RUN_ENRICH_EXISTING_CSV:
    existing_rows = read_csv_rows(INPUT_CSV)
    if REFRESH_CLASS_ANNOTATIONS:
        existing_rows = refresh_class_annotations(existing_rows)
    fieldnames = list(dict.fromkeys(list(existing_rows[0].keys()) + ENRICHMENT_FIELDNAMES)) if existing_rows else FIELDNAMES + ENRICHMENT_FIELDNAMES
    enriched_rows = enrich_rows(existing_rows)
    write_csv_rows(OUTPUT_CSV, enriched_rows, fieldnames)
    print(f"Wrote {len(enriched_rows)} enriched rows -> {OUTPUT_CSV}")
else:
    print("Skipped optional enrichment-only pass. Set RUN_ENRICH_EXISTING_CSV = True to run it.")


## 10. BioPortal Hierarchy Module Configuration

This section uses the merged class-search CSV from the earlier cells. It only extracts rows that are backed by BioPortal, because the hierarchy endpoint is BioPortal-specific.

In [ ]:
from collections import defaultdict

HIERARCHY_DEFAULT_CSV = SCOPING_REVIEW_DIR / "disease_terms_review_enriched.csv"
HIERARCHY_DEFAULT_OUT_DIR = SCOPING_REVIEW_DIR / "bioportal_hierarchy_modules"
HIERARCHY_DEFAULT_MERGED = SCOPING_REVIEW_DIR / "disease_terms_bioportal_hierarchy.ttl"
HIERARCHY_DEFAULT_MANIFEST = SCOPING_REVIEW_DIR / "bioportal_hierarchy_modules_manifest.csv"
HIERARCHY_DEFAULT_CACHE_DIR = Path("/tmp/bioportal_hierarchy_cache")

OLS_HIERARCHY_DEFAULT_OUT_DIR = SCOPING_REVIEW_DIR / "ols_hierarchy_modules"
OLS_HIERARCHY_DEFAULT_MERGED = SCOPING_REVIEW_DIR / "disease_terms_ols_hierarchy.ttl"
OLS_HIERARCHY_DEFAULT_MANIFEST = SCOPING_REVIEW_DIR / "ols_hierarchy_modules_manifest.csv"
OLS_HIERARCHY_DEFAULT_CACHE_DIR = Path("/tmp/ols_hierarchy_cache")

OBO_HIERARCHY_DEFAULT_OUT_DIR = SCOPING_REVIEW_DIR / "obo_direct_hierarchy_modules"
OBO_HIERARCHY_DEFAULT_MERGED = SCOPING_REVIEW_DIR / "disease_terms_obo_direct_hierarchy.ttl"
OBO_HIERARCHY_DEFAULT_MANIFEST = SCOPING_REVIEW_DIR / "obo_direct_hierarchy_modules_manifest.csv"
OBO_HIERARCHY_DEFAULT_CACHE_DIR = Path("/tmp/obo_direct_hierarchy_cache")

RDF_TYPE = "http://www.w3.org/1999/02/22-rdf-syntax-ns#type"
OWL_CLASS = "http://www.w3.org/2002/07/owl#Class"
RDFS_LABEL = "http://www.w3.org/2000/01/rdf-schema#label"
RDFS_COMMENT = "http://www.w3.org/2000/01/rdf-schema#comment"
RDFS_SUBCLASS_OF = "http://www.w3.org/2000/01/rdf-schema#subClassOf"
SKOS_ALT_LABEL = "http://www.w3.org/2004/02/skos/core#altLabel"
IAO_DEFINITION = "http://purl.obolibrary.org/obo/IAO_0000115"
IAO_EDITOR_NOTE = "http://purl.obolibrary.org/obo/IAO_0000116"
DCTERMS_SOURCE = "http://purl.org/dc/terms/source"
DCTERMS_CREATED = "http://purl.org/dc/terms/created"
OWL_VERSION_INFO = "http://www.w3.org/2002/07/owl#versionInfo"

HIERARCHY_DEFINITION_KEYS = {
    "definition",
    "description",
    "iao_0000115",
    "obo:iao_0000115",
    IAO_DEFINITION.lower(),
    "http://www.w3.org/2004/02/skos/core#definition",
}
HIERARCHY_COMMENT_KEYS = {
    "comment",
    "comments",
    "rdfs:comment",
    RDFS_COMMENT.lower(),
    "skos:note",
    "skos:scopenote",
    "scopeNote",
    "iao_0000116",
    IAO_EDITOR_NOTE.lower(),
}
HIERARCHY_SYNONYM_KEYS = {
    "synonym",
    "synonyms",
    "hasExactSynonym",
    "hasRelatedSynonym",
    "http://www.geneontology.org/formats/oboInOwl#hasExactSynonym".lower(),
    "http://www.geneontology.org/formats/oboInOwl#hasRelatedSynonym".lower(),
}


## 11. BioPortal Hierarchy API Helpers

In [ ]:
def hierarchy_unique(values: list[str]) -> list[str]:
    seen = set()
    out = []
    for value in values:
        if value not in seen:
            out.append(value)
            seen.add(value)
    return out


def hierarchy_extract_properties(record: dict[str, Any], wanted_keys: set[str]) -> list[str]:
    values: list[str] = []
    properties = record.get("properties")
    if not isinstance(properties, dict):
        return values
    for key, value in properties.items():
        if annotation_key_matches(str(key), wanted_keys):
            values.extend(clean_list(value))
    return values


def hierarchy_cache_path(cache_dir: Path, url: str, params: dict[str, Any]) -> Path:
    serialised = json.dumps([url, sorted(params.items())], sort_keys=True, default=str)
    digest = hashlib.sha256(serialised.encode("utf-8")).hexdigest()
    cache_dir.mkdir(parents=True, exist_ok=True)
    return cache_dir / f"{digest}.json"


def hierarchy_get_json(url: str, params: dict[str, Any], cache_dir: Path, retries: int = 3) -> Any | None:
    path = hierarchy_cache_path(cache_dir, url, params)
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    for attempt in range(retries):
        try:
            time.sleep(DELAY)
            response = requests.get(url, params=params, timeout=60)
            if response.status_code == 429:
                wait = int(response.headers.get("Retry-After", 10))
                print(f"  [rate limit] waiting {wait}s", file=sys.stderr)
                time.sleep(wait)
                continue
            if response.status_code in {400, 404}:
                print(f"  [warn] GET {url} returned {response.status_code}; skipping", file=sys.stderr)
                return None
            response.raise_for_status()
            data = response.json()
            path.write_text(json.dumps(data), encoding="utf-8")
            return data
        except Exception as exc:
            print(f"  [warn] GET {url} attempt {attempt + 1} failed: {exc}", file=sys.stderr)
            time.sleep(2 ** attempt)
    return None


def hierarchy_encoded_iri(iri: str) -> str:
    return urllib.parse.quote(iri, safe="")


def hierarchy_class_url(ontology: str, iri: str, suffix: str = "") -> str:
    base = f"{BIOPORTAL_BASE_URL}/ontologies/{ontology}/classes/{hierarchy_encoded_iri(iri)}"
    return f"{base}/{suffix}" if suffix else base


def hierarchy_fetch_class(ontology: str, iri: str, apikey: str, cache_dir: Path) -> dict[str, Any]:
    data = hierarchy_get_json(
        hierarchy_class_url(ontology, iri),
        {"apikey": apikey, "include": "prefLabel,definition,synonym,notation,properties"},
        cache_dir,
    )
    return data if isinstance(data, dict) else {}


def hierarchy_fetch_ancestors(ontology: str, iri: str, apikey: str, cache_dir: Path) -> list[dict[str, Any]]:
    data = hierarchy_get_json(
        hierarchy_class_url(ontology, iri, "ancestors"),
        {"apikey": apikey, "include": "prefLabel,definition,synonym,notation,properties"},
        cache_dir,
    )
    return data if isinstance(data, list) else []


def hierarchy_fetch_parents(ontology: str, iri: str, apikey: str, cache_dir: Path) -> list[dict[str, Any]]:
    data = hierarchy_get_json(
        hierarchy_class_url(ontology, iri, "parents"),
        {"apikey": apikey, "include": "prefLabel,definition,synonym,notation,properties"},
        cache_dir,
    )
    return data if isinstance(data, list) else []


## 12. Turtle Serialization

In [ ]:
def hierarchy_triples_for_class(record: dict[str, Any]) -> set[tuple[str, str, str, bool]]:
    iri = record.get("@id", "")
    if not iri:
        return set()

    triples: set[tuple[str, str, str, bool]] = {(iri, RDF_TYPE, OWL_CLASS, True)}
    labels = clean_list(record.get("prefLabel"))
    definitions = clean_list(record.get("definition")) + hierarchy_extract_properties(record, HIERARCHY_DEFINITION_KEYS)
    comments = hierarchy_extract_properties(record, HIERARCHY_COMMENT_KEYS)
    synonyms = clean_list(record.get("synonym")) + hierarchy_extract_properties(record, HIERARCHY_SYNONYM_KEYS)

    for label in hierarchy_unique(labels):
        triples.add((iri, RDFS_LABEL, label, False))
    for definition in hierarchy_unique(definitions):
        triples.add((iri, IAO_DEFINITION, definition, False))
    for comment in hierarchy_unique(comments):
        triples.add((iri, RDFS_COMMENT, comment, False))
    for synonym in hierarchy_unique(synonyms):
        triples.add((iri, SKOS_ALT_LABEL, synonym, False))
    return triples


def hierarchy_ttl_escape_literal(value: str) -> str:
    return json.dumps(value, ensure_ascii=False)


def hierarchy_ttl_iri(iri: str) -> str:
    return f"<{iri}>"


def hierarchy_write_ttl(
    path: Path,
    ontology_iri: str,
    triples: set[tuple[str, str, str, bool]],
    *,
    source_label: str = "BioPortal API hierarchy extraction",
    ontology_versions: str = "",
    retrieved_at: str = "",
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        handle.write("@prefix owl: <http://www.w3.org/2002/07/owl#> .\n")
        handle.write("@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .\n")
        handle.write("@prefix skos: <http://www.w3.org/2004/02/skos/core#> .\n")
        handle.write("@prefix iao: <http://purl.obolibrary.org/obo/IAO_> .\n")
        handle.write("@prefix dcterms: <http://purl.org/dc/terms/> .\n\n")
        handle.write(f"{hierarchy_ttl_iri(ontology_iri)} a owl:Ontology ;\n")
        handle.write(f"    dcterms:source {hierarchy_ttl_escape_literal(source_label)}")
        if ontology_versions:
            handle.write(f" ;\n    owl:versionInfo {hierarchy_ttl_escape_literal(ontology_versions)}")
        if retrieved_at:
            handle.write(f" ;\n    dcterms:created {hierarchy_ttl_escape_literal(retrieved_at)}")
        handle.write(f" ;\n    rdfs:comment {hierarchy_ttl_escape_literal('Contains review target classes, BioPortal ancestors, direct superclass edges, and common annotations only. Full source OWL axioms are not preserved.')} .\n\n")
        for subject, predicate, obj, obj_is_iri in sorted(triples):
            object_text = hierarchy_ttl_iri(obj) if obj_is_iri else hierarchy_ttl_escape_literal(obj)
            handle.write(f"{hierarchy_ttl_iri(subject)} {hierarchy_ttl_iri(predicate)} {object_text} .\n")


def hierarchy_safe_name(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", value.strip()).lower()


## 13. Read Search Results For Module Extraction

In [ ]:
def hierarchy_source_names(row: dict[str, str]) -> set[str]:
    source_text = row.get("source_databases", "")
    return {part.strip().casefold() for part in source_text.split(";") if part.strip()}


def hierarchy_parse_source_records(row: dict[str, str]) -> list[dict[str, Any]]:
    raw = row.get("source_records", "") or "[]"
    try:
        records = json.loads(raw)
    except json.JSONDecodeError:
        return []
    return records if isinstance(records, list) else []


def hierarchy_source_specific_rows(row: dict[str, str], source_name: str) -> list[dict[str, str]]:
    wanted = source_name.casefold()
    matches: list[dict[str, str]] = []
    for record in hierarchy_parse_source_records(row):
        if str(record.get("source", "")).casefold() != wanted:
            continue
        expanded = dict(row)
        expanded["ontology"] = str(record.get("ontology") or row.get("ontology", ""))
        expanded["hierarchy_source_database"] = source_name
        expanded["hierarchy_source_url"] = str(record.get("source_url") or "")
        expanded["hierarchy_source_retrieved_at"] = str(record.get("retrieved_at") or "")
        expanded["hierarchy_source_ontology_version"] = str(record.get("ontology_version") or "")
        if expanded["hierarchy_source_ontology_version"]:
            expanded["ontology_versions"] = f"{source_name}: {expanded['hierarchy_source_ontology_version']}"
        matches.append(expanded)

    if matches:
        return matches
    if wanted in hierarchy_source_names(row):
        fallback = dict(row)
        fallback["hierarchy_source_database"] = source_name
        fallback["hierarchy_source_url"] = ""
        fallback["hierarchy_source_retrieved_at"] = ""
        fallback["hierarchy_source_ontology_version"] = ""
        return [fallback]
    return []


def read_source_hierarchy_rows(
    path: Path,
    source_name: str,
    *,
    only_candidates: bool = True,
) -> list[dict[str, str]]:
    rows = []
    with path.open(newline="", encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            if only_candidates and row.get("ontology_exclusion_status") == "excluded_preasserted":
                continue
            if not row.get("class_iri"):
                continue
            for expanded in hierarchy_source_specific_rows(row, source_name):
                if expanded.get("ontology") and expanded.get("class_iri"):
                    rows.append(expanded)
    return rows


def read_bioportal_hierarchy_rows(
    path: Path,
    *,
    only_candidates: bool = True,
    only_bioportal_rows: bool = True,
) -> list[dict[str, str]]:
    if only_bioportal_rows:
        return read_source_hierarchy_rows(path, "BioPortal", only_candidates=only_candidates)
    rows = []
    with path.open(newline="", encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            if only_candidates and row.get("ontology_exclusion_status") == "excluded_preasserted":
                continue
            if row.get("ontology") and row.get("class_iri"):
                rows.append(row)
    return rows


## 14. Extract BioPortal Hierarchy Modules

In [ ]:
def extract_bioportal_hierarchy_modules(
    rows: list[dict[str, str]],
    *,
    apikey: str,
    out_dir: Path,
    merged_path: Path,
    manifest_path: Path,
    cache_dir: Path,
    source_csv: Path,
    limit_rows: int = 0,
) -> None:
    if limit_rows:
        rows = rows[:limit_rows]

    grouped: dict[str, list[dict[str, str]]] = defaultdict(list)
    for row in rows:
        grouped[row["ontology"]].append(row)

    all_triples: set[tuple[str, str, str, bool]] = set()
    manifest_rows = []
    run_retrieved_at = now_iso()

    for ontology_index, (ontology, ontology_rows) in enumerate(sorted(grouped.items()), start=1):
        print(f"[{ontology_index}/{len(grouped)}] {ontology}: {len(ontology_rows)} target rows", file=sys.stderr)
        module_triples: set[tuple[str, str, str, bool]] = set()
        nodes: dict[str, dict[str, Any]] = {}
        target_iris = {row["class_iri"] for row in ontology_rows}
        ontology_versions = join_values(row.get("ontology_versions", "") for row in ontology_rows)

        for row in ontology_rows:
            iri = row["class_iri"]
            target_record = hierarchy_fetch_class(ontology, iri, apikey, cache_dir)
            if target_record and target_record.get("@id"):
                nodes[target_record["@id"]] = target_record
            ancestors = hierarchy_fetch_ancestors(ontology, iri, apikey, cache_dir)
            for ancestor in ancestors:
                if ancestor.get("@id"):
                    nodes[ancestor["@id"]] = ancestor

        node_iris = set(nodes)
        edge_count = 0
        for child_iri in sorted(node_iris):
            parents = hierarchy_fetch_parents(ontology, child_iri, apikey, cache_dir)
            for parent in parents:
                parent_iri = parent.get("@id", "")
                if parent_iri and parent_iri in node_iris:
                    module_triples.add((child_iri, RDFS_SUBCLASS_OF, parent_iri, True))
                    edge_count += 1
                    nodes.setdefault(parent_iri, parent)

        for record in nodes.values():
            module_triples.update(hierarchy_triples_for_class(record))
        for target_iri in target_iris:
            module_triples.add((target_iri, DCTERMS_SOURCE, f"review target in {ontology} from {source_csv.name}", False))

        module_path = out_dir / f"{hierarchy_safe_name(ontology)}_hierarchy.ttl"
        ontology_iri = f"https://example.org/disease-review/modules/{hierarchy_safe_name(ontology)}_hierarchy"
        hierarchy_write_ttl(
            module_path,
            ontology_iri,
            module_triples,
            ontology_versions=ontology_versions,
            retrieved_at=run_retrieved_at,
        )
        all_triples.update(module_triples)
        manifest_rows.append(
            {
                "ontology": ontology,
                "target_rows": len(ontology_rows),
                "target_iris": len(target_iris),
                "classes_written": len(node_iris),
                "subclass_edges": edge_count,
                "ontology_versions": ontology_versions,
                "retrieval_source_database": "BioPortal",
                "retrieved_at": run_retrieved_at,
                "source_csv": str(source_csv),
                "module_path": str(module_path),
            }
        )

    hierarchy_write_ttl(
        merged_path,
        "https://example.org/disease-review/bioportal-hierarchy",
        all_triples,
        retrieved_at=run_retrieved_at,
    )
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    with manifest_path.open("w", newline="", encoding="utf-8") as handle:
        fieldnames = [
            "ontology",
            "target_rows",
            "target_iris",
            "classes_written",
            "subclass_edges",
            "ontology_versions",
            "retrieval_source_database",
            "retrieved_at",
            "source_csv",
            "module_path",
        ]
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(manifest_rows)
    print(f"Wrote {len(manifest_rows)} modules -> {out_dir}", file=sys.stderr)
    print(f"Wrote merged hierarchy -> {merged_path}", file=sys.stderr)
    print(f"Wrote manifest -> {manifest_path}", file=sys.stderr)


## 15. Select Rows For Hierarchy Extraction

In [ ]:
# Run parameters - edit these before running the hierarchy extraction.
HIERARCHY_CSV_PATH = HIERARCHY_DEFAULT_CSV
HIERARCHY_OUT_DIR = HIERARCHY_DEFAULT_OUT_DIR
HIERARCHY_MERGED_PATH = HIERARCHY_DEFAULT_MERGED
HIERARCHY_MANIFEST_PATH = HIERARCHY_DEFAULT_MANIFEST
HIERARCHY_CACHE_DIR = HIERARCHY_DEFAULT_CACHE_DIR
HIERARCHY_INCLUDE_PREEXCLUDED = False
HIERARCHY_ONLY_BIOPORTAL_ROWS = True
HIERARCHY_LIMIT_ROWS = 0  # set to e.g. 10 for a small smoke test

hierarchy_review_rows = read_bioportal_hierarchy_rows(
    Path(HIERARCHY_CSV_PATH),
    only_candidates=not HIERARCHY_INCLUDE_PREEXCLUDED,
    only_bioportal_rows=HIERARCHY_ONLY_BIOPORTAL_ROWS,
)
if HIERARCHY_LIMIT_ROWS:
    hierarchy_review_rows = hierarchy_review_rows[:HIERARCHY_LIMIT_ROWS]

hierarchy_by_ontology = defaultdict(int)
for row in hierarchy_review_rows:
    hierarchy_by_ontology[row["ontology"]] += 1

print(f"BioPortal source-specific target rows: {len(hierarchy_review_rows)}")
print(f"Ontologies: {len(hierarchy_by_ontology)}")
list(sorted(hierarchy_by_ontology.items()))[:20]


## 16. Run Hierarchy Extraction

In [ ]:
# Extract BioPortal hierarchy modules.
# This can take a while for the full candidate set because it calls BioPortal for classes, ancestors, and parent edges.
extract_bioportal_hierarchy_modules(
    hierarchy_review_rows,
    apikey=load_bioportal_api_key(),
    out_dir=Path(HIERARCHY_OUT_DIR),
    merged_path=Path(HIERARCHY_MERGED_PATH),
    manifest_path=Path(HIERARCHY_MANIFEST_PATH),
    cache_dir=Path(HIERARCHY_CACHE_DIR),
    source_csv=Path(HIERARCHY_CSV_PATH),
    limit_rows=0,
)


## 17. Inspect Hierarchy Manifest

In [ ]:
# Inspect manifest summary after extraction.
if Path(HIERARCHY_MANIFEST_PATH).exists():
    with Path(HIERARCHY_MANIFEST_PATH).open(newline="", encoding="utf-8") as handle:
        hierarchy_manifest_rows = list(csv.DictReader(handle))
    print(f"Modules: {len(hierarchy_manifest_rows)}")
    print(f"Classes written: {sum(int(r['classes_written']) for r in hierarchy_manifest_rows)}")
    print(f"Subclass edges: {sum(int(r['subclass_edges']) for r in hierarchy_manifest_rows)}")
    hierarchy_manifest_rows[:10]
else:
    print("Manifest does not exist yet. Run the hierarchy extraction cell first.")


## 18. OLS Hierarchy Module Helpers

OLS hierarchy extraction is source-specific: merged rows are expanded through `source_records`, then each OLS ontology occurrence is walked through OLS parent endpoints.

In [ ]:
def hierarchy_ols_term_record(ontology: str, iri: str, base_url: str) -> dict[str, Any]:
    detail = fetch_ols_term_details(ontology, iri, base_url)
    if not detail:
        return {"@id": iri, "prefLabel": "", "definition": [], "synonym": [], "properties": {}}

    annotations = detail.get("annotation") if isinstance(detail.get("annotation"), dict) else {}
    properties: dict[str, Any] = dict(annotations)
    for key in ["comment", "comments"]:
        if detail.get(key):
            properties[key] = detail.get(key)

    return {
        "@id": detail.get("iri") or iri,
        "prefLabel": detail.get("label") or detail.get("prefLabel") or "",
        "definition": clean_list(detail.get("description")) + extract_annotation_values(detail, DEFINITION_KEYS),
        "synonym": clean_list(detail.get("synonym")) + clean_list(detail.get("exact_synonyms")),
        "properties": properties,
    }


def extract_ols_hierarchy_modules(
    rows: list[dict[str, str]],
    *,
    base_url: str,
    out_dir: Path,
    merged_path: Path,
    manifest_path: Path,
    source_csv: Path,
    max_depth: int = 25,
    limit_rows: int = 0,
) -> None:
    if limit_rows:
        rows = rows[:limit_rows]

    base_url = normalise_ols_base_url(base_url)
    grouped: dict[str, list[dict[str, str]]] = defaultdict(list)
    for row in rows:
        grouped[row["ontology"].lower()].append(row)

    all_triples: set[tuple[str, str, str, bool]] = set()
    manifest_rows = []
    run_retrieved_at = now_iso()

    for ontology_index, (ontology, ontology_rows) in enumerate(sorted(grouped.items()), start=1):
        print(f"[OLS {ontology_index}/{len(grouped)}] {ontology}: {len(ontology_rows)} target rows", file=sys.stderr)
        module_triples: set[tuple[str, str, str, bool]] = set()
        nodes: dict[str, dict[str, Any]] = {}
        edge_count = 0
        target_iris = {row["class_iri"] for row in ontology_rows}
        ontology_versions = join_values([fetch_ols_ontology_version(ontology, base_url)] + [row.get("hierarchy_source_ontology_version", "") for row in ontology_rows])

        queue: list[tuple[str, int]] = [(iri, 0) for iri in sorted(target_iris)]
        seen_depth: dict[str, int] = {}
        while queue:
            iri, depth = queue.pop(0)
            if iri in seen_depth and seen_depth[iri] <= depth:
                continue
            seen_depth[iri] = depth
            record = hierarchy_ols_term_record(ontology, iri, base_url)
            if record.get("@id"):
                nodes[record["@id"]] = record
            if depth >= max_depth:
                continue
            parents = fetch_ols_parents(ontology, iri, base_url)
            for parent in parents:
                parent_iri = parent.get("@id", "")
                if not parent_iri:
                    continue
                module_triples.add((iri, RDFS_SUBCLASS_OF, parent_iri, True))
                edge_count += 1
                if parent_iri not in nodes and parent.get("prefLabel"):
                    nodes[parent_iri] = {"@id": parent_iri, "prefLabel": parent.get("prefLabel", ""), "definition": [], "synonym": [], "properties": {}}
                queue.append((parent_iri, depth + 1))

        for row in ontology_rows:
            if row["class_iri"] not in nodes:
                nodes[row["class_iri"]] = {
                    "@id": row["class_iri"],
                    "prefLabel": row.get("preferred_label", ""),
                    "definition": clean_list(row.get("definition", "").split(";")),
                    "synonym": clean_list(row.get("synonyms", "").split(";")),
                    "properties": {RDFS_COMMENT: clean_list(row.get("comments", "").split(";"))},
                }
            module_triples.add((row["class_iri"], DCTERMS_SOURCE, f"review target in OLS {ontology} from {source_csv.name}", False))

        for record in nodes.values():
            module_triples.update(hierarchy_triples_for_class(record))

        module_path = out_dir / f"{hierarchy_safe_name(ontology)}_ols_hierarchy.ttl"
        ontology_iri = f"https://example.org/disease-review/modules/{hierarchy_safe_name(ontology)}_ols_hierarchy"
        hierarchy_write_ttl(
            module_path,
            ontology_iri,
            module_triples,
            source_label="OLS API hierarchy extraction",
            ontology_versions=ontology_versions,
            retrieved_at=run_retrieved_at,
        )
        all_triples.update(module_triples)
        manifest_rows.append(
            {
                "ontology": ontology,
                "target_rows": len(ontology_rows),
                "target_iris": len(target_iris),
                "classes_written": len(nodes),
                "subclass_edges": edge_count,
                "ontology_versions": ontology_versions,
                "retrieval_source_database": "OLS",
                "retrieved_at": run_retrieved_at,
                "source_csv": str(source_csv),
                "module_path": str(module_path),
            }
        )

    hierarchy_write_ttl(
        merged_path,
        "https://example.org/disease-review/ols-hierarchy",
        all_triples,
        source_label="OLS API hierarchy extraction",
        retrieved_at=run_retrieved_at,
    )
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    with manifest_path.open("w", newline="", encoding="utf-8") as handle:
        fieldnames = ["ontology", "target_rows", "target_iris", "classes_written", "subclass_edges", "ontology_versions", "retrieval_source_database", "retrieved_at", "source_csv", "module_path"]
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(manifest_rows)
    print(f"Wrote {len(manifest_rows)} OLS modules -> {out_dir}", file=sys.stderr)


## 19. OBO Foundry Direct Hierarchy Module Helpers

OBO direct extraction parses the source file found during the OBO Foundry scan. It walks explicit named superclass links available in that downloaded file; anonymous restrictions and imported axioms are intentionally not reconstructed.

In [ ]:
def hierarchy_parse_obo_graph(text: str) -> tuple[str, dict[str, dict[str, Any]], dict[str, list[str]]]:
    header = parse_obo_header(text)
    version = header.get("data-version") or header.get("date") or header.get("format-version") or ""
    nodes: dict[str, dict[str, Any]] = {}
    edges: dict[str, list[str]] = defaultdict(list)
    current_kind = ""
    current: dict[str, list[str]] = {}

    def flush_current() -> None:
        if current_kind != "Term" or not current:
            return
        if first_clean(current.get("is_obsolete")).lower() == "true":
            return
        curie = first_clean(current.get("id"))
        if not curie:
            return
        iri = obo_curie_to_iri(curie)
        definition = ""
        if current.get("def"):
            match = re.search(r'"(.*?)"', current["def"][0])
            definition = match.group(1) if match else current["def"][0]
        synonyms = []
        for synonym_line in current.get("synonym", []):
            match = re.search(r'"(.*?)"', synonym_line)
            if match:
                synonyms.append(match.group(1))
        nodes[iri] = {
            "@id": iri,
            "prefLabel": first_clean(current.get("name")),
            "definition": clean_list(definition),
            "synonym": synonyms,
            "properties": {RDFS_COMMENT: current.get("comment", [])},
        }
        for parent_line in current.get("is_a", []):
            parent_curie = parent_line.split()[0]
            if parent_curie:
                edges[iri].append(obo_curie_to_iri(parent_curie))

    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line or line.startswith("!"):
            continue
        if line.startswith("[") and line.endswith("]"):
            flush_current()
            current_kind = line.strip("[]")
            current = {}
            continue
        if current_kind and ":" in line:
            key, value = line.split(":", 1)
            current.setdefault(key.strip(), []).append(value.strip())
    flush_current()
    return version, nodes, edges


def hierarchy_parse_owl_xml_graph(text: str) -> tuple[str, dict[str, dict[str, Any]], dict[str, list[str]]]:
    try:
        root = ET.fromstring(text.encode("utf-8"))
    except ET.ParseError as exc:
        print(f"  [warn] OWL/XML parse failed: {exc}", file=sys.stderr)
        return "", {}, {}

    rdf_about = "{http://www.w3.org/1999/02/22-rdf-syntax-ns#}about"
    rdf_resource = "{http://www.w3.org/1999/02/22-rdf-syntax-ns#}resource"
    owl_version_iri = "{http://www.w3.org/2002/07/owl#}versionIRI"
    owl_version_info = "{http://www.w3.org/2002/07/owl#}versionInfo"

    version = ""
    for ontology_element in root.iter("{http://www.w3.org/2002/07/owl#}Ontology"):
        for child in ontology_element:
            if child.tag == owl_version_iri and child.attrib.get(rdf_resource):
                version = child.attrib[rdf_resource]
                break
            if child.tag == owl_version_info and child.text:
                version = child.text.strip()
                break
        if version:
            break

    nodes: dict[str, dict[str, Any]] = {}
    edges: dict[str, list[str]] = defaultdict(list)
    for element in root.iter("{http://www.w3.org/2002/07/owl#}Class"):
        iri = element.attrib.get(rdf_about, "")
        if not iri:
            continue
        definitions = child_texts(element, {"IAO_0000115", "definition"})
        comments = child_texts(element, {"comment"})
        synonyms = []
        for child in element:
            name = local_name(child.tag)
            if "Synonym" in name and child.text and child.text.strip():
                synonyms.append(child.text.strip())
            if local_name(child.tag) == "subClassOf" and child.attrib.get(rdf_resource):
                edges[iri].append(child.attrib[rdf_resource])
        nodes[iri] = {
            "@id": iri,
            "prefLabel": first_clean(child_texts(element, {"label"})),
            "definition": definitions,
            "synonym": synonyms,
            "properties": {RDFS_COMMENT: comments},
        }
    return version, nodes, edges


def hierarchy_parse_ontology_graph(text: str) -> tuple[str, dict[str, dict[str, Any]], dict[str, list[str]]]:
    stripped = text.lstrip()
    if stripped.startswith("format-version:") or "\n[Term]" in stripped[:5000]:
        return hierarchy_parse_obo_graph(text)
    if stripped.startswith("<?xml") or stripped.startswith("<rdf:RDF") or stripped.startswith("<Ontology"):
        return hierarchy_parse_owl_xml_graph(text)
    print("  [warn] ontology format not supported for OBO direct hierarchy extraction", file=sys.stderr)
    return "", {}, {}


def hierarchy_obo_source_url(row: dict[str, str]) -> str:
    if row.get("hierarchy_source_url"):
        return row["hierarchy_source_url"]
    for part in str(row.get("source_urls", "")).split(";"):
        part = part.strip()
        if part.casefold().startswith("obo foundry direct:"):
            return part.split(":", 1)[1].strip()
    return ""


def extract_obo_direct_hierarchy_modules(
    rows: list[dict[str, str]],
    *,
    out_dir: Path,
    merged_path: Path,
    manifest_path: Path,
    source_csv: Path,
    max_depth: int = 50,
    limit_rows: int = 0,
) -> None:
    if limit_rows:
        rows = rows[:limit_rows]

    grouped: dict[tuple[str, str], list[dict[str, str]]] = defaultdict(list)
    for row in rows:
        source_url = hierarchy_obo_source_url(row)
        if source_url:
            grouped[(row["ontology"].lower(), source_url)].append(row)

    all_triples: set[tuple[str, str, str, bool]] = set()
    manifest_rows = []
    run_retrieved_at = now_iso()

    for group_index, ((ontology, source_url), ontology_rows) in enumerate(sorted(grouped.items()), start=1):
        print(f"[OBO {group_index}/{len(grouped)}] {ontology}: {len(ontology_rows)} target rows", file=sys.stderr)
        text = get_text(source_url, namespace="obo-direct-hierarchy")
        if not text:
            continue
        parsed_version, source_nodes, source_edges = hierarchy_parse_ontology_graph(text)
        module_triples: set[tuple[str, str, str, bool]] = set()
        nodes: dict[str, dict[str, Any]] = {}
        edge_count = 0
        target_iris = {row["class_iri"] for row in ontology_rows}
        ontology_versions = join_values([parsed_version] + [row.get("hierarchy_source_ontology_version", "") for row in ontology_rows])

        queue: list[tuple[str, int]] = [(iri, 0) for iri in sorted(target_iris)]
        seen_depth: dict[str, int] = {}
        while queue:
            iri, depth = queue.pop(0)
            if iri in seen_depth and seen_depth[iri] <= depth:
                continue
            seen_depth[iri] = depth
            if iri in source_nodes:
                nodes[iri] = source_nodes[iri]
            elif iri not in nodes:
                nodes[iri] = {"@id": iri, "prefLabel": "", "definition": [], "synonym": [], "properties": {}}
            if depth >= max_depth:
                continue
            for parent_iri in source_edges.get(iri, []):
                module_triples.add((iri, RDFS_SUBCLASS_OF, parent_iri, True))
                edge_count += 1
                if parent_iri in source_nodes:
                    nodes[parent_iri] = source_nodes[parent_iri]
                elif parent_iri not in nodes:
                    nodes[parent_iri] = {"@id": parent_iri, "prefLabel": "", "definition": [], "synonym": [], "properties": {}}
                queue.append((parent_iri, depth + 1))

        for row in ontology_rows:
            if row["class_iri"] not in nodes or not nodes[row["class_iri"]].get("prefLabel"):
                nodes[row["class_iri"]] = {
                    "@id": row["class_iri"],
                    "prefLabel": row.get("preferred_label", ""),
                    "definition": clean_list(row.get("definition", "").split(";")),
                    "synonym": clean_list(row.get("synonyms", "").split(";")),
                    "properties": {RDFS_COMMENT: clean_list(row.get("comments", "").split(";"))},
                }
            module_triples.add((row["class_iri"], DCTERMS_SOURCE, f"review target in OBO Foundry Direct {ontology} from {source_csv.name}", False))

        for record in nodes.values():
            module_triples.update(hierarchy_triples_for_class(record))

        module_path = out_dir / f"{hierarchy_safe_name(ontology)}_obo_direct_hierarchy.ttl"
        ontology_iri = f"https://example.org/disease-review/modules/{hierarchy_safe_name(ontology)}_obo_direct_hierarchy"
        hierarchy_write_ttl(
            module_path,
            ontology_iri,
            module_triples,
            source_label=f"OBO Foundry Direct hierarchy extraction from {source_url}",
            ontology_versions=ontology_versions,
            retrieved_at=run_retrieved_at,
        )
        all_triples.update(module_triples)
        manifest_rows.append(
            {
                "ontology": ontology,
                "target_rows": len(ontology_rows),
                "target_iris": len(target_iris),
                "classes_written": len(nodes),
                "subclass_edges": edge_count,
                "ontology_versions": ontology_versions,
                "retrieval_source_database": "OBO Foundry Direct",
                "retrieved_at": run_retrieved_at,
                "source_csv": str(source_csv),
                "source_url": source_url,
                "module_path": str(module_path),
            }
        )

    hierarchy_write_ttl(
        merged_path,
        "https://example.org/disease-review/obo-direct-hierarchy",
        all_triples,
        source_label="OBO Foundry Direct hierarchy extraction",
        retrieved_at=run_retrieved_at,
    )
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    with manifest_path.open("w", newline="", encoding="utf-8") as handle:
        fieldnames = ["ontology", "target_rows", "target_iris", "classes_written", "subclass_edges", "ontology_versions", "retrieval_source_database", "retrieved_at", "source_csv", "source_url", "module_path"]
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(manifest_rows)
    print(f"Wrote {len(manifest_rows)} OBO direct modules -> {out_dir}", file=sys.stderr)


## 20. Select OLS And OBO Rows

In [ ]:
# Optional source-specific hierarchy rows beyond BioPortal.
OLS_HIERARCHY_CSV_PATH = HIERARCHY_DEFAULT_CSV
OLS_HIERARCHY_OUT_DIR = OLS_HIERARCHY_DEFAULT_OUT_DIR
OLS_HIERARCHY_MERGED_PATH = OLS_HIERARCHY_DEFAULT_MERGED
OLS_HIERARCHY_MANIFEST_PATH = OLS_HIERARCHY_DEFAULT_MANIFEST
OLS_HIERARCHY_BASE_URL = OLS_BASE_URL_NOTEBOOK
OLS_HIERARCHY_MAX_DEPTH = 25

OBO_HIERARCHY_CSV_PATH = HIERARCHY_DEFAULT_CSV
OBO_HIERARCHY_OUT_DIR = OBO_HIERARCHY_DEFAULT_OUT_DIR
OBO_HIERARCHY_MERGED_PATH = OBO_HIERARCHY_DEFAULT_MERGED
OBO_HIERARCHY_MANIFEST_PATH = OBO_HIERARCHY_DEFAULT_MANIFEST
OBO_HIERARCHY_MAX_DEPTH = 50

ols_hierarchy_rows = read_source_hierarchy_rows(
    Path(OLS_HIERARCHY_CSV_PATH),
    "OLS",
    only_candidates=not HIERARCHY_INCLUDE_PREEXCLUDED,
)
obo_hierarchy_rows = read_source_hierarchy_rows(
    Path(OBO_HIERARCHY_CSV_PATH),
    "OBO Foundry Direct",
    only_candidates=not HIERARCHY_INCLUDE_PREEXCLUDED,
)

print(f"OLS source-specific target rows: {len(ols_hierarchy_rows)}")
print(f"OBO Foundry Direct source-specific target rows: {len(obo_hierarchy_rows)}")


## 21. Run OLS And OBO Hierarchy Extractions

In [ ]:
RUN_OLS_HIERARCHY_EXTRACTION = True
RUN_OBO_DIRECT_HIERARCHY_EXTRACTION = True

if RUN_OLS_HIERARCHY_EXTRACTION and ols_hierarchy_rows:
    extract_ols_hierarchy_modules(
        ols_hierarchy_rows,
        base_url=OLS_HIERARCHY_BASE_URL,
        out_dir=Path(OLS_HIERARCHY_OUT_DIR),
        merged_path=Path(OLS_HIERARCHY_MERGED_PATH),
        manifest_path=Path(OLS_HIERARCHY_MANIFEST_PATH),
        source_csv=Path(OLS_HIERARCHY_CSV_PATH),
        max_depth=OLS_HIERARCHY_MAX_DEPTH,
    )
else:
    print("Skipped OLS hierarchy extraction.")

if RUN_OBO_DIRECT_HIERARCHY_EXTRACTION and obo_hierarchy_rows:
    extract_obo_direct_hierarchy_modules(
        obo_hierarchy_rows,
        out_dir=Path(OBO_HIERARCHY_OUT_DIR),
        merged_path=Path(OBO_HIERARCHY_MERGED_PATH),
        manifest_path=Path(OBO_HIERARCHY_MANIFEST_PATH),
        source_csv=Path(OBO_HIERARCHY_CSV_PATH),
        max_depth=OBO_HIERARCHY_MAX_DEPTH,
    )
else:
    print("Skipped OBO direct hierarchy extraction.")


## 22. Inspect All Hierarchy Manifests

In [ ]:
for label, manifest_path in [
    ("BioPortal", HIERARCHY_MANIFEST_PATH),
    ("OLS", OLS_HIERARCHY_MANIFEST_PATH),
    ("OBO Foundry Direct", OBO_HIERARCHY_MANIFEST_PATH),
]:
    manifest_path = Path(manifest_path)
    if not manifest_path.exists():
        print(f"{label}: manifest not found")
        continue
    with manifest_path.open(newline="", encoding="utf-8") as handle:
        manifest_rows = list(csv.DictReader(handle))
    classes = sum(int(row.get("classes_written", 0) or 0) for row in manifest_rows)
    edges = sum(int(row.get("subclass_edges", 0) or 0) for row in manifest_rows)
    print(f"{label}: modules={len(manifest_rows)}, classes={classes}, subclass_edges={edges}")
